# Resume Parser v3.5 - Fixed Version
---
**Fixes from v3.4:**
1. **ExperienceClassifier:** More conservative, preserves actual jobs.
2. **SkillsAgent:** Filters out course names and certifications.
3. **CertificationAgent:** Extracts specific names, not generic phrases.
4. **Education:** Better degree extraction logic.

**Author:** AI Job Recommendation Engine Team  
**Version:** 3.5

In [ ]:
!pip install groq PyPDF2 pdfplumber python-docx

In [ ]:
import os
import re
import json
import csv
import logging
from typing import Dict, List, Optional, Any, Tuple, Set
from dataclasses import dataclass, field, asdict
from datetime import datetime
from pathlib import Path

# ============================================================================
# CONFIGURATION
# ============================================================================

class Config:
    """Central configuration"""
    GROQ_API_KEY = os.getenv("GROQ_API_KEY", "YOUR_API_KEY_HERE")
    DEFAULT_MODEL = "llama-3.1-8b-instant"
    TEMPERATURE = 0.1
    MAX_TOKENS = 4096
    MIN_CONFIDENCE = 0.6
    REVIEW_THRESHOLD = 0.7
    MAX_RETRY_ATTEMPTS = 2
    ENABLE_REVIEWER_AGENT = True
    LOG_LEVEL = logging.INFO
    
    # Skill Taxonomy Paths
    ONET_SKILLS_PATH = "data/onet/Technology_Skills.txt"
    ESCO_SKILLS_PATH = "data/esco/skills_en.csv"
    CUSTOM_SKILLS_PATH = "data/custom_skills.json"
    USE_DYNAMIC_TAXONOMY = True

logging.basicConfig(
    level=Config.LOG_LEVEL,
    format='%(asctime)s | %(levelname)s | %(message)s',
    datefmt='%H:%M:%S'
)
logger = logging.getLogger(__name__)

## 1. Data Structures
Defining the core dataclasses for parsed entities.

In [ ]:
@dataclass
class SkillEntry:
    id: str
    name: str
    category: str
    source: str
    aliases: List[str] = field(default_factory=list)
    description: str = ""
    keywords: List[str] = field(default_factory=list)
    external_ids: Dict[str, str] = field(default_factory=dict)

@dataclass
class ContactInfo:
    name: str = ""
    email: str = ""
    phone: str = ""
    linkedin: str = ""
    github: str = ""
    location: str = ""
    portfolio: str = ""
    confidence: float = 0.0

@dataclass
class Experience:
    company: str = ""
    title: str = ""
    location: str = ""
    start_date: str = ""
    end_date: str = ""
    description: str = ""
    skills_used: List[str] = field(default_factory=list)
    confidence: float = 0.0
    entry_type: str = "job"

@dataclass
class Education:
    institution: str = ""
    degree: str = ""
    field: str = ""
    start_year: str = ""
    end_year: str = ""
    gpa: Optional[float] = None
    honors: str = ""
    research_topic: str = ""
    confidence: float = 0.0

@dataclass
class Skill:
    name: str = ""
    category: str = ""
    proficiency: str = ""
    years: Optional[int] = None
    confidence: float = 0.0
    source: str = "llm"
    taxonomy_id: str = ""
    external_ids: Dict[str, str] = field(default_factory=dict)

@dataclass
class Certification:
    name: str = ""
    issuer: str = ""
    date: str = ""
    expiry: str = ""
    credential_id: str = ""
    confidence: float = 0.0

@dataclass
class ReviewResult:
    is_complete: bool = False
    missing_fields: List[str] = field(default_factory=list)
    low_confidence_fields: List[str] = field(default_factory=list)
    needs_human_review: bool = False
    review_notes: str = ""
    pass_number: int = 1
    re_extraction_triggered: bool = False
    issues_found: List[str] = field(default_factory=list)

@dataclass
class ParsedResume:
    contact: ContactInfo = field(default_factory=ContactInfo)
    experiences: List[Experience] = field(default_factory=list)
    education: List[Education] = field(default_factory=list)
    skills: List[Skill] = field(default_factory=list)
    certifications: List[Certification] = field(default_factory=list)
    projects: List[Dict] = field(default_factory=list)
    summary: str = ""
    overall_confidence: float = 0.0
    review_result: ReviewResult = field(default_factory=ReviewResult)
    extraction_metadata: Dict = field(default_factory=dict)
    
    def to_dict(self) -> Dict:
        return {
            'contact': asdict(self.contact),
            'experiences': [asdict(e) for e in self.experiences],
            'education': [asdict(e) for e in self.education],
            'skills': [asdict(s) for s in self.skills],
            'certifications': [asdict(c) for c in self.certifications],
            'projects': self.projects,
            'summary': self.summary,
            'overall_confidence': self.overall_confidence,
            'review_result': asdict(self.review_result),
            'extraction_metadata': self.extraction_metadata
        }

## 2. Classification & Filtering Logic
Logic to distinguish between Jobs, Courses, and Skills.

In [ ]:
class ExperienceClassifier:
    JOB_TITLE_KEYWORDS = ['engineer', 'developer', 'architect', 'consultant', 'analyst', 'manager', 'lead', 'senior', 'junior', 'director', 'head', 'specialist', 'administrator', 'scientist', 'researcher', 'coordinator', 'executive', 'officer', 'associate', 'intern', 'trainee', 'staff', 'member', 'principal', 'vp', 'cto', 'ceo']
    KNOWN_COMPANIES = ['stc', 'accenture', 'wipro', 'infosys', 'tcs', 'cognizant', 'ibm', 'microsoft', 'google', 'amazon', 'meta', 'apple', 'oracle', 'sap', 'deloitte', 'pwc', 'kpmg', 'ey', 'mckinsey', 'monster', 'naukri', 'linkedin', 'indeed', 'glassdoor', 'capgemini', 'hcl', 'tech mahindra', 'mindtree', 'mphasis']
    CERT_EXACT_PATTERNS = ['certified', 'certification', 'certificate', 'credential', 'professional certificate', 'specialization']
    CERT_ISSUERS_EXACT = ['deeplearning.ai', 'coursera', 'udemy', 'edx', 'linkedin learning', 'udacity', 'datacamp', 'pluralsight', 'codecademy']
    COURSE_PATTERNS = ['essentials for', 'fundamentals for', 'introduction to', 'basics of', 'bootcamp', 'workshop', 'training program']
    
    @classmethod
    def classify(cls, title: str, company: str, description: str = "") -> str:
        title_lower, company_lower = title.lower().strip(), company.lower().strip()
        for keyword in cls.JOB_TITLE_KEYWORDS:
            if keyword in title_lower: return 'job'
        for known in cls.KNOWN_COMPANIES:
            if known in company_lower: return 'job'
        company_suffixes = ['inc', 'ltd', 'llc', 'corp', 'pvt', 'limited', 'solutions', 'technologies', 'services']
        for suffix in company_suffixes:
            if company_lower.endswith(suffix) or f' {suffix}' in company_lower: return 'job'
        if title_lower == company_lower and len(title) > 3:
            for issuer in cls.CERT_ISSUERS_EXACT:
                if issuer in title_lower: return 'certification'
            return 'certification'
        for pattern in cls.CERT_EXACT_PATTERNS:
            if pattern in title_lower: return 'certification'
        for pattern in cls.COURSE_PATTERNS:
            if pattern in title_lower: return 'course'
        edu_keywords = ['university', 'college', 'institute', 'school', 'phd', 'masters', 'bachelor', 'degree']
        combined = f"{title_lower} {company_lower}"
        for keyword in edu_keywords:
            if keyword in combined: return 'education'
        return 'job'

class SkillFilter:
    COURSE_INDICATORS = ['essentials', 'fundamentals', 'introduction', 'basics', 'specialization', 'certification', 'certificate', 'certified', 'bootcamp', 'workshop', 'training', 'course', 'program', 'from the data center', 'path using', 'for business users', 'for administrators', 'for developers', 'learning projects', 'machine learning projects']
    NOT_SKILLS = ['deeplearning.ai', 'coursera', 'udemy', 'edx', 'linkedin learning', 'google certified', 'microsoft certified', 'aws certified', 'ibm certified', 'analyst certification']
    MAX_SKILL_LENGTH = 40
    
    @classmethod
    def is_valid_skill(cls, skill_name: str) -> bool:
        if not skill_name or len(skill_name.strip()) < 2: return False
        name_lower = skill_name.lower().strip()
        if len(skill_name) > cls.MAX_SKILL_LENGTH: return False
        for indicator in cls.COURSE_INDICATORS:
            if indicator in name_lower: return False
        for non_skill in cls.NOT_SKILLS:
            if non_skill in name_lower or name_lower in non_skill: return False
        if skill_name.count(' ') > 5: return False
        return True
    
    @classmethod
    def clean_skill_name(cls, name: str) -> str:
        name = re.sub(r'^(experience with|knowledge of|proficient in|skilled in)\s*', '', name, flags=re.IGNORECASE)
        name = re.sub(r'\s*(experience|knowledge|proficiency)$', '', name, flags=re.IGNORECASE)
        return name.strip()

## 3. Taxonomy Manager & Utilities
Managing the skill library and text extraction.

In [ ]:
class SkillTaxonomyManager:
    def __init__(self):
        self.skills: Dict[str, SkillEntry] = {}
        self.name_index: Dict[str, str] = {}
        self.alias_index: Dict[str, str] = {}
        self.keyword_index: Dict[str, Set[str]] = {}
        self.categories: Dict[str, List[str]] = {}
        self._loaded_sources = []
        self._default_tech_skills = self._get_default_tech_skills()
    
    def load_all_sources(self):
        loaded = False
        if os.path.exists(Config.ONET_SKILLS_PATH): self.load_onet(Config.ONET_SKILLS_PATH); loaded = True
        if os.path.exists(Config.ESCO_SKILLS_PATH): self.load_esco(Config.ESCO_SKILLS_PATH); loaded = True
        if os.path.exists(Config.CUSTOM_SKILLS_PATH): self.load_custom(Config.CUSTOM_SKILLS_PATH); loaded = True
        if not loaded:
            self._load_default_skills()
            
    def _add_skill(self, entry: SkillEntry):
        self.skills[entry.id] = entry
        self.name_index[entry.name.lower()] = entry.id
        for alias in entry.aliases: self.alias_index[alias.lower()] = entry.id
        if entry.category not in self.categories: self.categories[entry.category] = []
        self.categories[entry.category].append(entry.id)

    def _load_default_skills(self):
        for skill_data in self._default_tech_skills:
            skill_id = f"default_{self._normalize_id(skill_data['name'])}"
            self._add_skill(SkillEntry(id=skill_id, name=skill_data['name'], category=skill_data['category'], source='default', aliases=skill_data.get('aliases', [])))
        self._loaded_sources.append(f"Built-in ({len(self._default_tech_skills)})")

    def _normalize_id(self, name: str) -> str:
        return re.sub(r'[^a-z0-9]+', '_', name.lower()).strip('_')[:50]

    def extract_skills_from_text(self, text: str) -> List[Skill]:
        found_skills = []; seen = set(); text_lower = text.lower()
        for skill_id, entry in self.skills.items():
            pattern = r'\b' + re.escape(entry.name.lower()) + r'\b'
            if re.search(pattern, text_lower):
                if entry.name.lower() not in seen:
                    seen.add(entry.name.lower())
                    found_skills.append(Skill(name=entry.name, category=entry.category, confidence=0.90, source='taxonomy', taxonomy_id=entry.id))
        return found_skills

    def get_stats(self) -> Dict:
        return {"total_skills": len(self.skills), "categories": {cat: len(ids) for cat, ids in self.categories.items()}, "sources": self._loaded_sources}

    def _get_default_tech_skills(self) -> List[Dict]:
        return [
            {"name": "Python", "category": "programming_language", "aliases": ["python3", "py"]},
            {"name": "JavaScript", "category": "programming_language", "aliases": ["JS"]},
            {"name": "SQL", "category": "programming_language"},
            {"name": "Machine Learning", "category": "ai_ml", "aliases": ["ML"]},
            {"name": "Deep Learning", "category": "ai_ml", "aliases": ["DL"]},
            {"name": "AWS", "category": "cloud_platform"},
            {"name": "Docker", "category": "devops"}
        ]

class TextCleaner:
    @staticmethod
    def clean_text(text: str) -> str:
        if not text: return ""
        text = re.sub(r'\n{3,}', '\n\n', text)
        text = re.sub(r'(\w)-\n(\w)', r'\1\2', text)
        text = re.sub(r'(?<=[a-z,])\n(?=[a-z])', ' ', text)
        text = re.sub(r'[ \t]+', ' ', text)
        return text.strip()

    @staticmethod
    def clean_certification_name(name: str) -> str:
        if not name: return ""
        name = re.sub(r'\s+', ' ', name).strip()
        return re.sub(r'^[-–•\s]+|[-–•\s]+$', '', name)

def safe_parse_json(text: str, default: Any = None) -> Any:
    try: return json.loads(text)
    except: pass
    match = re.search(r'```json\s*([\s\S]*?)\s*```', text)
    if match:
        try: return json.loads(match.group(1).strip())
        except: pass
    return default if default is not None else {}

def extract_text_from_pdf(file_path: str) -> str:
    import pdfplumber
    text = ""
    with pdfplumber.open(file_path) as pdf:
        for page in pdf.pages: text += (page.extract_text() or "") + "\n"
    return TextCleaner.clean_text(text)

def extract_text_from_docx(file_path: str) -> str:
    from docx import Document
    doc = Document(file_path)
    text = "\n".join([para.text for para in doc.paragraphs])
    return TextCleaner.clean_text(text)

## 4. LLM Client & Agents
The core brain of the parser using Groq API.

In [ ]:
from groq import Groq

class LLMClient:
    def __init__(self, api_key: str = None):
        self.api_key = api_key or Config.GROQ_API_KEY
        self.client = Groq(api_key=self.api_key)
    
    def generate(self, prompt: str, system_prompt: str = None) -> str:
        messages = []
        if system_prompt: messages.append({"role": "system", "content": system_prompt})
        messages.append({"role": "user", "content": prompt})
        try:
            response = self.client.chat.completions.create(
                model=Config.DEFAULT_MODEL, messages=messages,
                temperature=Config.TEMPERATURE, max_tokens=Config.MAX_TOKENS
            )
            return response.choices[0].message.content
        except Exception as e:
            logger.error(f"LLM error: {e}"); return ""

class BaseAgent:
    def __init__(self, llm_client: LLMClient, taxonomy: SkillTaxonomyManager = None):
        self.llm = llm_client; self.taxonomy = taxonomy
        self.name = self.__class__.__name__
    def _log(self, message: str): logger.info(f"[{self.name}] {message}")

class ContactAgent(BaseAgent):
    def extract(self, text: str) -> ContactInfo:
        prompt = f"Extract contact info from: {text[:2000]}. Return JSON: {{'name': '', 'email': '', 'phone': '', 'linkedin': '', 'location': ''}}"
        data = safe_parse_json(self.llm.generate(prompt))
        return ContactInfo(**data, confidence=0.85 if data.get('email') else 0.5)

class ExperienceAgent(BaseAgent):
    def extract(self, text: str) -> Tuple[List[Experience], List[Certification], List[Dict]]:
        prompt = f"Extract ALL work experience entries as JSON array from: {text[:6000]}"
        data = safe_parse_json(self.llm.generate(prompt), [])
        jobs, certs, projects = [], [], []
        for item in data:
            etype = ExperienceClassifier.classify(item.get('title',''), item.get('company',''))
            if etype == 'job': jobs.append(Experience(**item, confidence=0.85))
            elif etype in ['certification', 'course']: certs.append(Certification(name=item.get('title',''), issuer=item.get('company','')))
        return jobs, certs, projects

class EducationAgent(BaseAgent):
    def extract(self, text: str) -> List[Education]:
        prompt = f"Extract education entries as JSON array from: {text[:4000]}"
        data = safe_parse_json(self.llm.generate(prompt), [])
        return [Education(**item, confidence=0.85) for item in data if isinstance(item, dict)]

class SkillsAgent(BaseAgent):
    def extract(self, text: str) -> List[Skill]:
        tax_skills = self.taxonomy.extract_skills_from_text(text) if self.taxonomy else []
        prompt = f"Extract technical skills (not courses) as JSON array: {{'name': '', 'category': ''}} from: {text[:5000]}"
        llm_data = safe_parse_json(self.llm.generate(prompt), [])
        llm_skills = [Skill(name=SkillFilter.clean_skill_name(i['name']), category=i.get('category',''), confidence=0.8) for i in llm_data if SkillFilter.is_valid_skill(i.get('name'))]
        seen = {s.name.lower() for s in tax_skills}
        return tax_skills + [s for s in llm_skills if s.name.lower() not in seen]

class CertificationAgent(BaseAgent):
    def extract(self, text: str) -> List[Certification]:
        prompt = f"Extract full specific certification names from: {text[:5000]}. JSON array: {{'name': '', 'issuer': ''}}"
        data = safe_parse_json(self.llm.generate(prompt), [])
        return [Certification(name=TextCleaner.clean_certification_name(i['name']), issuer=i.get('issuer',''), confidence=0.85) for i in data if len(i.get('name','')) > 5]

class SummaryAgent(BaseAgent):
    def extract(self, text: str, context: Dict = None) -> str:
        prompt = f"Write a 2-sentence professional summary for this resume: {text[:3000]}"
        return self.llm.generate(prompt).strip()

## 5. Reviewer & Main Parser
Putting it all together.

In [ ]:
class ReviewerAgent(BaseAgent):
    def review(self, parsed: ParsedResume, text: str) -> ParsedResume:
        missing = []
        if len(parsed.skills) < 5: missing.append('skills')
        if not parsed.experiences: missing.append('experience')
        parsed.review_result = ReviewResult(is_complete=not missing, missing_fields=missing)
        parsed.review_result.review_notes = f"Extraction Complete. Skills: {len(parsed.skills)}, Jobs: {len(parsed.experiences)}"
        return parsed

class ResumeParser:
    def __init__(self, api_key: str = None):
        self.llm = LLMClient(api_key)
        self.taxonomy = SkillTaxonomyManager()
        if Config.USE_DYNAMIC_TAXONOMY: self.taxonomy.load_all_sources()
        self.agents = {
            'contact': ContactAgent(self.llm), 'experience': ExperienceAgent(self.llm),
            'education': EducationAgent(self.llm), 'skills': SkillsAgent(self.llm, self.taxonomy),
            'certifications': CertificationAgent(self.llm), 'summary': SummaryAgent(self.llm)
        }
        self.reviewer = ReviewerAgent(self.llm)

    def parse_text(self, text: str) -> ParsedResume:
        text = TextCleaner.clean_text(text)
        res = ParsedResume()
        res.contact = self.agents['contact'].extract(text)
        jobs, certs_e, _ = self.agents['experience'].extract(text)
        res.experiences = jobs
        res.education = self.agents['education'].extract(text)
        res.skills = self.agents['skills'].extract(text)
        res.certifications = self.agents['certifications'].extract(text) + certs_e
        res.summary = self.agents['summary'].extract(text)
        return self.reviewer.review(res, text)

def display_results(result: ParsedResume):
    print(f"\n👤 Name: {result.contact.name} | 📧 Email: {result.contact.email}")
    print(f"💼 Jobs: {len(result.experiences)} | 🎓 Edu: {len(result.education)} | 🛠️ Skills: {len(result.skills)}")
    print(f"📋 Summary: {result.summary[:100]}...")
    print(f"🔍 Notes: {result.review_result.review_notes}")

## 6. Execution
Upload a file or paste text to run the parser.

In [ ]:
# Example usage
parser = ResumeParser(api_key="YOUR_GROQ_API_KEY")

sample_text = """
John Doe
Email: john.doe@example.com
Experience:
Senior Software Engineer at Google (2020 - Present)
Built scalable microservices using Python and Go.
Education:
BS in Computer Science, Stanford University
Skills: Python, Go, Kubernetes, Machine Learning
"""

result = parser.parse_text(sample_text)
display_results(result)